# Electron Microscopy Data Bank (EMDB) — Data Ingestion

**EMDB** (Electron Microscopy Data Bank) is a public repository for cryo-electron microscopy density maps, tomograms, and subtomogram averages. It is managed by PDBe at the European Bioinformatics Institute (EBI) and is one of the ELIXIR Core Data Resources. As of 2024 EMDB holds tens of thousands of entries, growing rapidly alongside advances in cryo-EM instrumentation and software.

Key data types stored in EMDB:

| Data type | Description |
|---|---|
| **Single-particle density maps** | 3-D Coulomb potential maps reconstructed from thousands of identical particle images; typically 1–5 Å resolution |
| **Tomograms** | 3-D reconstructions of unique cellular volumes; lower resolution (10–50 Å) but preserves in-situ context |
| **Subtomogram averages** | Averages of picked particles within tomograms; bridges the resolution gap between the two types above |
| **2-D crystallography maps** | Projection maps from ordered 2-D protein crystals embedded in lipid bilayers |

Key metadata fields per entry:

| Field | Description |
|---|---|
| `emdb_id` | Unique accession, e.g. `EMD-1273` |
| `title` | Descriptive title of the structure |
| `method` | Imaging modality (`singleParticle`, `tomography`, `subtomogramAveraging`, etc.) |
| `resolution_angstrom` | Final reconstruction resolution in Ångströms |
| `organism` | Source organism(s) of the imaged sample |
| `deposition_date` | Date the entry was deposited |
| `fitted_pdb_ids` | PDB accessions for atomic models fitted into the density map |

**API base:** `https://www.ebi.ac.uk/emdb/api/`

**Reference:** Lawson et al. (2021), *Nucleic Acids Research*, EMDB: Electron Microscopy Data Bank. https://doi.org/10.1093/nar/gkab1213

# TODO

* [x] **Ingest data**
    * [x] Connect to EMDB REST API and confirm access with a single entry fetch (EMD-1273)
    * [x] Search EMDB for a topic (ribosome), page through results, cache to `data/emdb_ribosome.json`
    * [x] Parse search results into a Polars DataFrame with correct dtypes
    * [x] Fetch overall database release statistics from the `/release/` endpoint
    * [x] Print DataFrame shape, dtypes, and head
* [ ] **Explore and clean**
    * [ ] Summarize entry counts by method, organism, and year
    * [ ] Inspect resolution distributions per method
    * [ ] Identify entries with fitted PDB models vs. map-only entries
* [ ] **Analysis**
    * [ ] Compare resolution trends over deposition years
    * [ ] Identify the most-studied organisms and molecular assemblies
    * [ ] Correlate resolution with microscope type and detector technology
* [ ] **Visualization**
    * [ ] Resolution histogram split by method (single-particle vs. tomography)
    * [ ] Timeline of EMDB depositions coloured by method
    * [ ] Scatter plot: resolution vs. deposition year with trend lines
* [ ] **Statistical analysis**
    * [ ] Discuss error in cryo-EM resolution estimates (FSC-based, gold-standard FSC)
    * [ ] Explain the Fourier Shell Correlation framework and its statistical interpretation
    * [ ] Multiple hypothesis considerations when screening structural databases at scale

In [ ]:
import requests
import time
import json
from pathlib import Path

import polars as pl

## 1. Ingest Data

### 1.1 Connect to EMDB API and Fetch a Single Entry

In [ ]:
EMDB_BASE = "https://www.ebi.ac.uk/emdb/api"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


def emdb_get(endpoint: str, params: dict = None) -> dict | list:
    """
    Send a GET request to the EMDB REST API.

    Parameters
    ----------
    endpoint : str
        API path relative to EMDB_BASE (e.g. "entry/EMD-1273").
    params : dict, optional
        Query parameters appended to the URL.

    Returns
    -------
    dict or list
        Parsed JSON response; single-entry endpoints return a dict,
        search endpoints return a list.
    """
    url = f"{EMDB_BASE}/{endpoint}"
    resp = requests.get(url, params=params or {}, timeout=30)
    resp.raise_for_status()
    time.sleep(0.5)  # polite delay — EBI fair-use policy
    return resp.json()


# ── Connectivity check: fetch EMD-1273 (a well-known tomogram of the immunological synapse) ──
entry = emdb_get("entry/EMD-1273")

# Navigate the nested structure to extract key fields
title = entry.get("admin", {}).get("title", "N/A")

# Resolution lives inside structure_determination_list (may be absent for tomograms)
sd_list = (
    entry.get("structure_determination_list", {})
         .get("structure_determination", [{}])
)
resolution_raw = (
    sd_list[0]
    .get("image_processing", [{}])[0]
    .get("final_reconstruction", {})
    .get("resolution", {})
    .get("valueOf_")
)
method = sd_list[0].get("method", "N/A")

# Organisms can appear in multiple supramolecules; collect all unique names
supramolecules = (
    entry.get("sample", {})
         .get("supramolecule_list", {})
         .get("supramolecule", [])
)
organisms = list({
    src.get("organism", {}).get("valueOf_")
    for sm in supramolecules
    for src in (sm.get("natural_source") or [])
    if src.get("organism", {}).get("valueOf_")
})

print(f"Title      : {title}")
print(f"Method     : {method}")
print(f"Resolution : {resolution_raw} Å")
print(f"Organism(s): {', '.join(organisms) if organisms else 'N/A'}")

### 1.2 Search EMDB and Cache Results

In [ ]:
SEARCH_QUERY = "ribosome"
SEARCH_CACHE = DATA_DIR / "emdb_ribosome.json"
PAGE_SIZE = 100  # entries per request


def fetch_emdb_search(
    query: str,
    cache_path: Path,
    page_size: int = PAGE_SIZE,
) -> list[dict]:
    """
    Search EMDB for a keyword and page through all results.

    The search endpoint returns a JSON array (no pagination envelope).
    We iterate by incrementing `start` until the returned page is smaller
    than `rows`, signalling the final page.

    Parameters
    ----------
    query : str
        Free-text search term, e.g. "ribosome".
    cache_path : Path
        Where to write/read the cached JSON array.
    page_size : int
        Number of entries per API request (max ~100).

    Returns
    -------
    list[dict]
        All matching EMDB entry records as raw dicts.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    all_entries: list[dict] = []
    start = 0

    while True:
        batch = emdb_get(
            "search/",
            params={"q": query, "wt": "json", "rows": page_size, "start": start},
        )
        # The API returns a plain list; an empty list signals end of results
        if not isinstance(batch, list) or len(batch) == 0:
            break
        all_entries.extend(batch)
        print(f"  Fetched {len(all_entries)} entries so far ...", end="\r")
        if len(batch) < page_size:
            # Last page — fewer results than requested means no more pages
            break
        start += page_size
        time.sleep(0.5)  # polite delay between pages

    print(f"\nTotal entries retrieved: {len(all_entries)}")
    cache_path.write_text(json.dumps(all_entries))
    print(f"Cached to: {cache_path}")
    return all_entries


results_raw = fetch_emdb_search(SEARCH_QUERY, SEARCH_CACHE)
print(f"Records in memory: {len(results_raw)}")

### 1.3 Parse into a Polars DataFrame

In [ ]:
def flatten_emdb_entry(e: dict) -> dict:
    """
    Flatten a raw EMDB entry dict into a single row-friendly record.

    Nested or list-valued fields are collapsed to scalars or pipe-separated
    strings so the result can be placed directly into a Polars DataFrame.

    Parameters
    ----------
    e : dict
        Raw entry record from the EMDB search or entry API.

    Returns
    -------
    dict
        Flat dict with the following keys:
        emdb_id, title, resolution_angstrom, method, organism,
        deposition_date, fitted_pdb_ids.
    """
    # ── structure determination block (first element) ────────────────────────
    sd_list = (
        e.get("structure_determination_list", {})
         .get("structure_determination", [{}])
    )
    sd = sd_list[0] if sd_list else {}

    # Resolution: buried inside image_processing → final_reconstruction
    ip_list = sd.get("image_processing", [{}])
    resolution_str = (
        ip_list[0]
        .get("final_reconstruction", {})
        .get("resolution", {})
        .get("valueOf_")
    ) if ip_list else None

    try:
        resolution = float(resolution_str) if resolution_str else None
    except (ValueError, TypeError):
        resolution = None

    # ── organisms: union across all supramolecules ───────────────────────────
    supramolecules = (
        e.get("sample", {})
         .get("supramolecule_list", {})
         .get("supramolecule", [])
    )
    organisms = list({
        src.get("organism", {}).get("valueOf_")
        for sm in supramolecules
        for src in (sm.get("natural_source") or [])
        if src.get("organism", {}).get("valueOf_")
    })

    # ── fitted PDB IDs ───────────────────────────────────────────────────────
    pdb_refs = (
        e.get("crossreferences", {})
         .get("pdb_list", {})
         .get("pdb_reference", [])
    )
    pdb_ids = [ref.get("pdb_id") for ref in pdb_refs if ref.get("pdb_id")]

    # ── deposition date: strip time component if present ────────────────────
    raw_date = e.get("admin", {}).get("key_dates", {}).get("deposition")
    deposition_date = raw_date[:10] if isinstance(raw_date, str) else None

    return {
        "emdb_id":             e.get("emdb_id"),
        "title":               e.get("admin", {}).get("title"),
        "resolution_angstrom": resolution,
        "method":              sd.get("method"),
        "organism":            " | ".join(organisms) if organisms else None,
        "deposition_date":     deposition_date,
        "fitted_pdb_ids":      " | ".join(pdb_ids) if pdb_ids else None,
    }


# ── Build the DataFrame ───────────────────────────────────────────────────────
rows = [flatten_emdb_entry(e) for e in results_raw]

emdb_df = pl.DataFrame(rows).with_columns(
    # Cast resolution to Float32 — sub-Ångström precision is never needed here
    pl.col("resolution_angstrom").cast(pl.Float32, strict=False),
    # Parse deposition_date as a proper Date type
    pl.col("deposition_date").str.to_date("%Y-%m-%d", strict=False),
)

print(f"Shape  : {emdb_df.shape}")
print(f"\nDtypes :")
print(emdb_df.schema)
print()
emdb_df.head(10)

### 1.4 Fetch Overall EMDB Release Statistics

In [ ]:
def fetch_release_stats() -> dict:
    """
    Fetch high-level database statistics from the EMDB release endpoint.

    The /release/ endpoint returns a summary of entry counts broken down
    by method type and other metadata. Useful as a quick sanity check
    and for contextualising the size of any subset we query.

    Returns
    -------
    dict
        Raw release stats dict from the EMDB API.
    """
    return emdb_get("release/")


release = fetch_release_stats()

# Pretty-print the top-level keys and their values
print("EMDB release statistics")
print("=" * 40)
for key, value in release.items():
    # Skip deeply nested sub-dicts — print them separately
    if not isinstance(value, dict):
        print(f"  {key:<30}: {value}")

# If there is a nested breakdown, print it
for key, value in release.items():
    if isinstance(value, dict):
        print(f"\n  {key}:")
        for k2, v2 in value.items():
            print(f"    {k2:<28}: {v2}")

### 1.5 DataFrame Summary

In [ ]:
# ── Shape and schema ─────────────────────────────────────────────────────────
print(f"Shape  : {emdb_df.shape[0]} rows × {emdb_df.shape[1]} columns")
print(f"\nSchema :")
for col, dtype in emdb_df.schema.items():
    print(f"  {col:<25} {dtype}")

# ── Null counts per column ───────────────────────────────────────────────────
print("\nNull counts per column:")
print(emdb_df.null_count())

# ── Descriptive statistics for numeric column ────────────────────────────────
print("\nResolution statistics (Å):")
print(emdb_df.select("resolution_angstrom").describe())

# ── Method breakdown ─────────────────────────────────────────────────────────
print("\nMethod breakdown:")
print(
    emdb_df.group_by("method")
           .agg(pl.len().alias("count"))
           .sort("count", descending=True)
)

# ── Head ─────────────────────────────────────────────────────────────────────
print("\nFirst 10 rows:")
emdb_df.head(10)